# 47 — Winding Number and Topological Sectors Under FIM

**INSIGHT FROM SPO:** V48 validated winding number conservation on a ring.
SPO suggested using winding number as Trotter error indicator.

**DEEPER QUESTION nobody asked:** Does FIM create or destroy topological defects?

In 2D XY models, BKT transition = vortex-antivortex unbinding.
NB43 showed SCPN is BKT. So there should be VORTEX-LIKE defects.
On a ring (1D), the analogue is the WINDING NUMBER q.

- Below K_c: vortices (q ≠ 0) proliferate
- Above K_c: vortices bind (q → 0 sector dominates)
- FIM: does it suppress vortices? Stabilise them? Change the BKT unbinding?

Also: The quantum ground state has a definite winding number.
Does FIM change which q sector has lowest energy?

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27

N = 16
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]


def fim_gradient_all(phases):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases + np.pi) % (2 * np.pi) - np.pi
    sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)
    return (1.0 / n) * np.sin(phase_diff) * sensitivity


def winding_number(theta):
    """Winding number on a ring: q = (1/2π) Σ Δθ_{i,i+1} (mod 2π)."""
    N = len(theta)
    total = 0
    for i in range(N):
        j = (i + 1) % N
        dtheta = (theta[j] - theta[i] + np.pi) % (2 * np.pi) - np.pi
        total += dtheta
    return round(total / (2 * np.pi))


def count_defects(theta):
    """Count phase defects (consecutive pairs with |Δθ| > π/2)."""
    N = len(theta)
    defects = 0
    for i in range(N):
        j = (i + 1) % N
        dtheta = abs((theta[j] - theta[i] + np.pi) % (2 * np.pi) - np.pi)
        if dtheta > np.pi / 2:
            defects += 1
    return defects


print("Ready.")

In [ ]:
def simulate_topology(K_scale, lam, dt=0.02, T=200.0, noise=0.05, seed=42):
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)

    q_history = []
    defect_history = []
    R_history = []

    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K * np.sin(diff), axis=1) / N
        dphi = omega + coupling
        if lam > 0:
            dphi += lam * fim_gradient_all(theta)
        theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2 * np.pi)

        if s % 50 == 0:
            q_history.append(winding_number(theta))
            defect_history.append(count_defects(theta))
            R_history.append(float(np.abs(np.mean(np.exp(1j * theta)))))

    return {
        "R_mean": float(np.mean(R_history[-len(R_history) // 4 :])),
        "q_final": q_history[-1],
        "q_std": float(np.std(q_history[-len(q_history) // 4 :])),
        "defects_mean": float(np.mean(defect_history[-len(defect_history) // 4 :])),
        "q_changes": sum(1 for i in range(1, len(q_history)) if q_history[i] != q_history[i - 1]),
    }


# Sweep K and λ
print("=== TOPOLOGICAL DEFECTS vs COUPLING AND FIM ===")
print(f"{'K':>6} {'λ':>4} {'R':>6} {'q':>4} {'σ_q':>6} {'defects':>8} {'q_changes':>10}")

topo_data = []
for K_scale in [2, 5, 8, 10, 12, 16, 20]:
    for lam in [0, 1, 5]:
        results = [simulate_topology(K_scale, lam, seed=s) for s in range(3)]
        R = np.mean([r["R_mean"] for r in results])
        q = results[0]["q_final"]
        q_std = np.mean([r["q_std"] for r in results])
        defects = np.mean([r["defects_mean"] for r in results])
        q_ch = np.mean([r["q_changes"] for r in results])

        topo_data.append(
            {
                "K": K_scale,
                "lam": lam,
                "R": round(R, 3),
                "q_std": round(q_std, 3),
                "defects": round(defects, 1),
                "q_changes": round(q_ch, 1),
            }
        )
        print(
            f"{K_scale:6.0f} {lam:4.0f} {R:6.3f} {q:4d} {q_std:6.3f} {defects:8.1f} {q_ch:10.1f}"
        )

In [ ]:
# Analysis: does FIM suppress topological defects?
print("\n=== FIM EFFECT ON TOPOLOGICAL DEFECTS ===")
for K_scale in [5, 10, 16]:
    d0 = [d for d in topo_data if d["K"] == K_scale and d["lam"] == 0][0]
    d5 = [d for d in topo_data if d["K"] == K_scale and d["lam"] == 5][0]
    print(
        f"K={K_scale}: defects {d0['defects']:.1f}→{d5['defects']:.1f} "
        f"({'SUPPRESSED' if d5['defects'] < d0['defects'] - 0.5 else 'unchanged'}), "
        f"q_changes {d0['q_changes']:.0f}→{d5['q_changes']:.0f}"
    )

print("\n=== BKT INTERPRETATION ===")
print("Below K_c: many defects, frequent q changes (vortex proliferation)")
print("Above K_c: few defects, stable q (vortices bound)")
print("FIM: should suppress defects (pull toward uniform phase)")

# Save
output = {
    "experiment": "Winding number and topological defects under FIM",
    "N": N,
    "data": topo_data,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/topological_defects_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")